In [1]:
import os
from pydoc import text

os.environ['TF_CPP_MIN_LOG_LEVEL']='3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tensorflow.keras import layers, models, optimizers, losses, metrics, preprocessing
from tensorflow.keras.layers import TextVectorization
import re, string

In [2]:
train_data_path = "../data/imdb/train.csv"
test_data_path = "../data/imdb/test.csv"

MAX_WORDS = 10000
MAX_LEN = 200
BATCH_SIZE = 20


def split_line(line):
    arr = tf.strings.split(line, "\t")
    label = tf.expand_dims(tf.cast(tf.strings.to_number(arr[0]), tf.int32), axis=0)
    text = tf.expand_dims(arr[1], axis=0)
    return text, label


ds_train_raw = tf.data.TextLineDataset(filenames=[train_data_path])\
    .map(split_line, num_parallel_calls=tf.data.AUTOTUNE).shuffle(buffer_size=1000).batch(BATCH_SIZE)\
    .prefetch(tf.data.AUTOTUNE)

ds_test_raw = tf.data.TextLineDataset(filenames=[test_data_path])\
    .map(split_line, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE)\
    .prefetch(tf.data.AUTOTUNE)

def clean_text(text):
    lowercase = tf.strings.lower(text)
    stripped_html = tf.strings.regex_replace(lowercase, "<br/>", " ")
    cleaned_punctuation = tf.strings.regex_replace(stripped_html, "[%s]" % re.escape(string.punctuation), " ")
    return cleaned_punctuation


vectorize_layer = TextVectorization(
    standardize=clean_text,
    split="whitespace",
    max_tokens=MAX_WORDS - 1,
    output_mode="int",
    output_sequence_length=MAX_LEN,
)

ds_train = ds_train_raw.map(lambda text, label: (vectorize_layer(text), label)).prefetch(tf.data.AUTOTUNE)

ds_test = ds_test_raw.map(lambda text, label: (vectorize_layer(text), label)).prefetch(tf.data.AUTOTUNE)

In [4]:
tf.keras.backend.clear_session()

class CnnModel(models.Model):
    def __init__(self):
        super().__init__()

    def build(self, input_shape):
        self.embedding = layers.Embedding(MAX_WORDS, 7, input_length=MAX_LEN)
        self.conv_1 = layers.Conv1D(16, kernel_size=5, name="conv_1", activation="relu")
        self.pool_1 = layers.MaxPool1D(name="pool_1")
        self.conv_2 = layers.Conv1D(128, kernel_size=2, name="conv_2", activation="relu")
        self.pool_2 = layers.MaxPool1D(name="pool_2")
        self.flatten = layers.Flatten()
        self.dense = layers.Dense(1, activation="sigmoid")
        super().build(input_shape)

    def call(self, x):
        x = self.embedding(x)
        x = self.conv_1(x)
        x = self.pool_1(x)
        x = self.conv_2(x)
        x = self.pool_2(x)
        x = self.flatten(x)
        x = self.dense(x)
        return x

    def summary(self):
        x_input = layers.Input(shape=(MAX_LEN,))
        output = self.call(x_input)
        model = models.Model(inputs=x_input, outputs=output)
        return model.summary()


model = CnnModel()
model.build(input_shape=(None, MAX_LEN))
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 200, 7)         │        70,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1 (Conv1D)                 │ (None, 196, 16)        │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 98, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_2 (Conv1D)                 │ (None, 97, 128)        │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 48, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6144)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         6,145 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 80,945 (316.19 KB)

 Trainable params: 80,945 (316.19 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
@tf.function
def printbar():
    ts = tf.timestamp()
    today_ts = tf.timestamp() % (24 * 60 * 60)
    hour = tf.cast(today_ts // 3600 + 8, tf.int32) % tf.constant(24)
    minute = tf.cast((today_ts % 3600) // 60, tf.int32)
    second = tf.cast(tf.floor(today_ts % 60), tf.int32)
    def timeformat(m):
        if tf.strings.length(tf.strings.format("{}", m)) == 1:
            return tf.strings.format("0{}", m)
        else:
            return tf.strings.format("{}", m)
    timestring = tf.strings.join([timeformat(hour), timeformat(minute), timeformat(second)], separator=":")
    tf.print("===========" * 8 + timestring)

In [10]:
optimizer = optimizers.Nadam()
loss_func = losses.BinaryCrossentropy()

train_loss = metrics.Mean(name="train_loss")
train_metric = metrics.BinaryCrossentropy(name="train_accuracy")

valid_loss = metrics.Mean(name="valid_loss")
valid_metric = metrics.BinaryCrossentropy(name="valid_accuracy")

@tf.function
def train_step(model, features, labels):
    with tf.GradientTape() as tape:
        predictions = model(features, training=True)
        loss = loss_func(labels, predictions)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    train_loss.update_state(loss)
    train_metric.update_state(labels, predictions)

@tf.function
def valid_step(model, features, labels):
    predictions = model(features, training=False)
    batch_loss = loss_func(labels, predictions)
    valid_loss.update_state(batch_loss)
    valid_metric.update_state(labels, predictions)


def train_model(model, ds_train, ds_valid, epochs=6):
    for epoch in tf.range(epochs + 1):
        for features, labels in ds_train:
            train_step(model, features, labels)
        for features, labels in ds_valid:
            valid_step(model, features, labels)

        logs = "Epoch={}, Loss: {}, Acc: {}, Valid Loss: {}, Valid Acc: {}"

        if epoch % 1 == 0:
            printbar()
            tf.print(logs, (epoch, train_loss.result(), train_metric.result(), valid_loss.result(), valid_metric.result()))
            tf.print("")

        train_loss.reset_state()
        train_metric.reset_state()
        valid_loss.reset_state()
        valid_metric.reset_state()


train_model(model, ds_train, ds_test, epochs=6)

NotFoundError: {{function_node __wrapped__IteratorGetNext_output_types_2_device_/job:localhost/replica:0/task:0/device:CPU:0}} ../data/imdb/train.csv; No such file or directory [Op:IteratorGetNext] name: 